<a href="https://colab.research.google.com/github/Pigwen/hands-on-sft/blob/main/Chapter_3_Low_Rank_Adaptation_(LoRA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
from copy import deepcopy
from numpy.linalg import matrix_rank
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Low-Rank Adaptation in a Nutshell

In [2]:
from torch import nn

base_layer = nn.Linear(1024, 1024, bias=False)
base_layer.weight.shape, base_layer.weight.numel()

(torch.Size([1024, 1024]), 1048576)

In [3]:
import torch

torch.manual_seed(11)
rank = 8
layer_A = nn.Linear(base_layer.in_features, rank, bias=False)
layer_B = nn.Linear(rank, base_layer.out_features, bias=False)
layer_A, layer_B

(Linear(in_features=1024, out_features=8, bias=False),
 Linear(in_features=8, out_features=1024, bias=False))

In [4]:
layer_A.weight.numel(), layer_B.weight.numel()

(8192, 8192)

In [5]:
composite = layer_B.weight @ layer_A.weight
composite.shape, composite.numel()

(torch.Size([1024, 1024]), 1048576)

In [6]:
from numpy.linalg import matrix_rank

matrix_rank(composite.detach().numpy())

np.int64(8)

In [7]:
torch.manual_seed(19)
batch = torch.randn(1, 1024)
batch @ (base_layer.weight + layer_B.weight @ layer_A.weight).T

tensor([[-0.2229,  0.3248, -1.2755,  ..., -0.2956,  0.5090,  0.0235]],
       grad_fn=<MmBackward0>)

# The Road So Far

In [8]:
! pip install bitsandbytes

In [9]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

supported = torch.cuda.is_bf16_supported(including_emulation=False)
compute_dtype = torch.bfloat16 if supported else torch.float32
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

model_q4 = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m", device_map="auto", dtype=compute_dtype, quantization_config=nf4_config
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [11]:
def trainable_params(model):
  return [(name, param.dtype) for name, param in model.named_parameters() if param.requires_grad]

trainable_params(model_q4)

[('model.decoder.embed_tokens.weight', torch.float32),
 ('model.decoder.embed_positions.weight', torch.float32),
 ('model.decoder.layers.0.self_attn_layer_norm.weight', torch.float32),
 ('model.decoder.layers.0.self_attn_layer_norm.bias', torch.float32),
 ('model.decoder.layers.0.final_layer_norm.weight', torch.float32),
 ('model.decoder.layers.0.final_layer_norm.bias', torch.float32),
 ('model.decoder.layers.1.self_attn_layer_norm.weight', torch.float32),
 ('model.decoder.layers.1.self_attn_layer_norm.bias', torch.float32),
 ('model.decoder.layers.1.final_layer_norm.weight', torch.float32),
 ('model.decoder.layers.1.final_layer_norm.bias', torch.float32),
 ('model.decoder.layers.2.self_attn_layer_norm.weight', torch.float32),
 ('model.decoder.layers.2.self_attn_layer_norm.bias', torch.float32),
 ('model.decoder.layers.2.final_layer_norm.weight', torch.float32),
 ('model.decoder.layers.2.final_layer_norm.bias', torch.float32),
 ('model.decoder.layers.3.self_attn_layer_norm.weight', tor

## prepare_model_for_kbit_training()

在进行LoRA训练之前，应该对量化后的模型调用`prepare_model_for_kbit_training`方法，这个方法有以下作用：

* 冻结整个模型的参数
* 将所有非量化的16bit layer转换为FP32（包含norm层和head层）
* 生效gradient checkpointing以减少内存抖动（第五章详细介绍）
  * 通过`{'use_reentrant': False}`参数生效

In [14]:
from peft import prepare_model_for_kbit_training

prepared_model = prepare_model_for_kbit_training(model_q4,
                                                 use_gradient_checkpointing=True,
                                                 gradient_checkpointing_kwargs={'use_reentrant': False})
prepared_model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear4bit(in_features=1024, out_features=512, bias=False)
      (project_in): Linear4bit(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
          (

In [15]:
trainable_params(prepared_model)

[]

In [16]:
def params_of_dtype(model, dtype=torch.float32):
  return [name for name, param in model.named_parameters() if param.dtype == dtype]

params_of_dtype(prepared_model)

['model.decoder.embed_tokens.weight',
 'model.decoder.embed_positions.weight',
 'model.decoder.layers.0.self_attn.k_proj.bias',
 'model.decoder.layers.0.self_attn.v_proj.bias',
 'model.decoder.layers.0.self_attn.q_proj.bias',
 'model.decoder.layers.0.self_attn.out_proj.bias',
 'model.decoder.layers.0.self_attn_layer_norm.weight',
 'model.decoder.layers.0.self_attn_layer_norm.bias',
 'model.decoder.layers.0.fc1.bias',
 'model.decoder.layers.0.fc2.bias',
 'model.decoder.layers.0.final_layer_norm.weight',
 'model.decoder.layers.0.final_layer_norm.bias',
 'model.decoder.layers.1.self_attn.k_proj.bias',
 'model.decoder.layers.1.self_attn.v_proj.bias',
 'model.decoder.layers.1.self_attn.q_proj.bias',
 'model.decoder.layers.1.self_attn.out_proj.bias',
 'model.decoder.layers.1.self_attn_layer_norm.weight',
 'model.decoder.layers.1.self_attn_layer_norm.bias',
 'model.decoder.layers.1.fc1.bias',
 'model.decoder.layers.1.fc2.bias',
 'model.decoder.layers.1.final_layer_norm.weight',
 'model.decode

In [17]:
prepared_model.get_memory_footprint() / 1e6

264.15104

注意：

* 最好将非量化层转换为bf16，以减少内存占用
  * 第5章会详细介绍使用混合精度(with bf16)将非量化层转换到16bit(norm层除外)
* 保持norm层可训可能会改进模型的性能
  * 能够使用LoRA配置(modules_to_save参数）解封部分weights
* 对于新模型来说，还不用在这个阶段引入Gradient checkpointing
  * SFTTrainer会处理这个（第五章详细介绍）

# PEFT